# 06 · Prediction and saving

How to turn raw outputs into answers, and how to ship a model together with its preprocessing.

The first cell rebuilds the data from **02_data_preparation**, then we train a model.

In [1]:
import random
import pandas as pd

ITEMS = ["apple", "bread", "cheese", "dates", "eggs"]
PRICE = {"apple": 20.0, "bread": 45.0, "cheese": 180.0, "dates": 95.0, "eggs": 60.0}

def make_events(n_users=300, seed=0):
    """One row per purchase. Each shopper buys ITEMS in a fixed cycle, starting at a random item."""
    rng = random.Random(seed)
    rows = []
    for user in range(n_users):
        start = rng.randrange(len(ITEMS))
        for t in range(rng.randint(6, 12)):
            item = ITEMS[(start + t) % len(ITEMS)]
            rows.append({"user_id": user, "t": t, "item": item,
                         "price": round(PRICE[item] * rng.uniform(0.9, 1.1), 2),
                         "weekday": rng.randrange(7)})
    return pd.DataFrame(rows)

import math
import torch
from torch.utils.data import DataLoader
from next_sequence import (CoreDataset, CustomCollate, Feature, NextSequence,
                           NextSequenceConfig, TaskType, create_bow)

events = make_events()
users = (events.sort_values(["user_id", "t"]).groupby("user_id").agg(list)
               .reset_index().drop(columns="t").to_dict(orient="records"))

def cut(user, end):
    return {k: (v[:end] if isinstance(v, list) else v) for k, v in user.items()}

train_raw = [cut(u, -2) for u in users]      # target: 3rd-to-last event
val_raw = [cut(u, -1) for u in users]        # target: 2nd-to-last event
test_raw = [cut(u, None) for u in users]     # target: last event

bow = create_bow(u["item"] for u in train_raw)
log_prices = [math.log1p(p) for u in train_raw for p in u["price"]]
price_mean = sum(log_prices) / len(log_prices)
price_std = (sum((v - price_mean) ** 2 for v in log_prices) / len(log_prices)) ** 0.5

def transform(samples):
    return [{"item": [bow.get(i, bow["UNK"]) for i in u["item"]],
             "price": [(math.log1p(p) - price_mean) / price_std for p in u["price"]],
             "weekday": [d + 1 for d in u["weekday"]]} for u in samples]

train, val, test = transform(train_raw), transform(val_raw), transform(test_raw)

def make_func(dtype):
    def func(values, inference=False):
        xs = [torch.tensor(v, dtype=dtype) for v in values]
        if inference:
            return xs, None
        return [x[:-1] for x in xs], [v[-1] for v in values]
    return func

features = [
    Feature("item", TaskType.CLASSIFICATION, vocab_size=len(bow)),
    Feature("price", TaskType.REGRESSION, is_linear=True),
    Feature("weekday", TaskType.PASSTHROUGH, vocab_size=8),
]
funcs = {"item": make_func(torch.long), "price": make_func(torch.float), "weekday": make_func(torch.long)}
MAX_LEN = 16

def loader(samples, inference=False, shuffle=False, batch_size=32):
    collate = CustomCollate(features, "item", funcs, inference=inference, max_len=MAX_LEN)
    return DataLoader(CoreDataset(samples), batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

print(f"{len(users)} users | train/val/test samples: {len(train)}/{len(val)}/{len(test)} | vocab: {len(bow)}")

300 users | train/val/test samples: 300/300/300 | vocab: 7


In [2]:
config = NextSequenceConfig(features=features, max_len=MAX_LEN, emb_dim=32, attn_heads=2)
model = NextSequence(config, epochs=15, lr=3e-3, seed=0, verbose=False).fit(loader(train, shuffle=True), loader(val))
print(model)

NextSequence(features=['item', 'price', 'weekday'], epochs=15, lr=0.003, device='cuda', fitted)


## Two ways to predict

| Loader built with | Input | Use it to |
|---|---|---|
| `inference=False` | all but the last event (`Y` is ignored by `predict`) | **evaluate** against the known last event |
| `inference=True` | the full history | **predict the future**: the event after the history |

`predict` needs a loader with `shuffle=False` so row *i* of the output is sample *i*.

In [3]:
eval_out = model.predict(loader(test))
future_out = model.predict(loader(test, inference=True))
{name: tuple(t.shape) for name, t in eval_out.items()}

{'item': (300, 7), 'price': (300, 1)}

## Decoding classification: logits → items

`mask_pad=True` sets the PAD logit to `-inf`. Do the same for `UNK` if you never want to predict it.
Use `softmax` for probabilities and `topk` for several candidates.

In [4]:
out = model.predict(loader(test), mask_pad=True)
logits = out["item"].clone()
logits[:, bow["UNK"]] = float("-inf")

id2item = {i: t for t, i in bow.items()}
probs = logits.softmax(-1)
top_p, top_id = probs.topk(3, dim=-1)

pd.DataFrame({
    "history (end)": [u["item"][-4:-1] for u in test_raw[:5]],
    "true next": [u["item"][-1] for u in test_raw[:5]],
    "top 3": [[f"{id2item[i]} {p:.2f}" for i, p in zip(ids, ps)]
              for ids, ps in zip(top_id[:5].tolist(), top_p[:5].tolist())],
})

,history (end),true next,top 3
0,"[bread, cheese, dates]",eggs,"[eggs 0.98, dates 0.02, apple 0.00]"
1,"[dates, eggs, apple]",bread,"[apple 0.51, bread 0.48, dates 0.00]"
2,"[apple, bread, cheese]",dates,"[dates 0.99, cheese 0.00, bread 0.00]"
3,"[eggs, apple, bread]",cheese,"[cheese 0.99, dates 0.00, eggs 0.00]"
4,"[apple, bread, cheese]",dates,"[dates 0.99, cheese 0.00, bread 0.00]"


In [5]:
pred_item = [id2item[i] for i in logits.argmax(-1).tolist()]
accuracy = sum(p == u["item"][-1] for p, u in zip(pred_item, test_raw)) / len(test_raw)
print(f"next-item accuracy on the test split: {accuracy:.3f}")

next-item accuracy on the test split: 0.940


## Decoding regression: undo the scaling

The model predicts in the scale you trained on. Here that is standardized `log1p(price)`.

In [6]:
pred_price = torch.expm1(out["price"].squeeze(-1) * price_std + price_mean)
true_price = torch.tensor([u["price"][-1] for u in test_raw])
print(f"price MAE: {(pred_price - true_price).abs().mean():.2f}")
pd.DataFrame({"item": [u["item"][-1] for u in test_raw[:5]],
              "true price": true_price[:5].double().numpy().round(2), "predicted": pred_price[:5].double().numpy().round(2)})

price MAE: 7.83


,item,true price,predicted
0,eggs,57.72,67.38
1,bread,40.51,24.73
2,dates,102.13,97.23
3,cheese,173.41,178.90
4,dates,87.02,96.17


## Save with your preprocessing

The model can't decode its outputs without the vocabulary and scaler, so store them in the same
file with `metadata`. Keep it to plain Python types (dict, list, str, int, float, bool, None) and tensors.

In [7]:
model.save("outputs/shop_model.pt",
           metadata={"bow": bow, "price_mean": price_mean, "price_std": price_std, "max_len": MAX_LEN})

## Load it somewhere else

`device=None` picks CUDA if available. A model trained on a GPU also loads on a CPU-only machine.

In [8]:
restored = NextSequence.load("outputs/shop_model.pt", device="cpu")
meta = restored.metadata_
print(restored)
print("best epoch:", restored.best_epoch_, "| metadata:", list(meta))

NextSequence(features=['item', 'price', 'weekday'], epochs=15, lr=0.003, device='cpu', fitted)
best epoch: 15 | metadata: ['bow', 'price_mean', 'price_std', 'max_len']


## Predict for brand-new data

Transform new histories with the **saved** preprocessing, build an inference loader, predict.
`durian` was never seen in training, so it becomes `UNK`.

In [9]:
new_users = [
    {"item": ["apple", "bread", "cheese"], "price": [21.0, 44.0, 185.0], "weekday": [1, 2, 3]},
    {"item": ["dates", "durian", "apple"], "price": [97.0, 300.0, 19.0], "weekday": [4, 5, 6]},
]
bow2 = meta["bow"]
new_samples = [{"item": [bow2.get(i, bow2["UNK"]) for i in u["item"]],
                "price": [(math.log1p(p) - meta["price_mean"]) / meta["price_std"] for p in u["price"]],
                "weekday": [d + 1 for d in u["weekday"]]} for u in new_users]

new_out = restored.predict(loader(new_samples, inference=True), mask_pad=True)
new_logits = new_out["item"]
new_logits[:, bow2["UNK"]] = float("-inf")
id2 = {i: t for t, i in bow2.items()}
pd.DataFrame({
    "history": [u["item"] for u in new_users],
    "predicted next item": [id2[i] for i in new_logits.argmax(-1).tolist()],
    "predicted price": torch.expm1(new_out["price"].squeeze(-1) * meta["price_std"] + meta["price_mean"]).double().numpy().round(2),
})

,history,predicted next item,predicted price
0,"[apple, bread, cheese]",dates,103.84
1,"[dates, durian, apple]",bread,51.17
